# Graph search: dynamic programming and UCS (Uniform Cost Search)

_Artificial Intelligence (CS221)_

**Don't redo work. Remember states you've solved, and always expand the cheapest one next.**

Tree search can visit the same state many times. Wasteful.

     Graph search remembers states it has already handled, so it never repeats them.

---

This notebook builds **Uniform Cost Search** from scratch, one step at a time. We'll keep a priority queue ordered by cost-so-far, always expand the cheapest frontier node, and mark states "settled" so we never process them twice. Run each cell, read the note above it, then watch UCS find the cheapest route on a small map. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

We'll find the cheapest path from `S` to `G` on a small weighted graph. We build it in three steps: (1) turn the edge list into an adjacency map, (2) set up the priority queue UCS uses, (3) run the UCS loop that always expands the cheapest node first.

### Step 1 — Build the graph as an adjacency list

The graph is a list of weighted edges `(u, v, w)`. To search efficiently we convert it into an **adjacency list**: a dictionary mapping each node to its neighbours and the cost to reach them.

The edges are **undirected**, so each edge is added in both directions — you can travel either way along it.

In [ ]:
import heapq

# Weighted edges: (node_u, node_v, cost).
edges = [
    ("S", "A", 1), ("S", "B", 4), ("A", "C", 2), ("A", "B", 2),
    ("B", "D", 1), ("C", "G", 3), ("D", "G", 2), ("C", "D", 1),
]

# Build an undirected adjacency list: node -> list of (neighbour, cost).
nbr = {}
for u, v, w in edges:
    nbr.setdefault(u, []).append((v, w))
    nbr.setdefault(v, []).append((u, w))

print("nodes:", sorted(nbr))
print("neighbours of S:", nbr["S"])

### Step 2 — Set up the priority queue

UCS uses a **min-heap** (priority queue) keyed by the total cost spent so far. We seed it with the start node at cost `0`, carrying along the path taken to reach it.

The `settled` set records states whose cheapest cost is already finalized — this is the "don't redo work" idea that makes it *graph* search rather than tree search.

In [ ]:
start, goal = "S", "G"

# Priority queue entries are (past_cost, node, path_so_far).
# heapq orders by the first element, so the cheapest node always pops first.
pq = [(0, start, [start])]

# States whose cheapest cost is already finalized; never re-expand these.
settled = set()

### Step 3 — Run the UCS loop

Repeatedly pop the cheapest frontier node. If it's already **settled**, skip it (we've found a cheaper route before). Otherwise settle it — at this moment its popped cost is guaranteed minimal, the key UCS invariant.

If it's the goal, we're done. Otherwise we push each unsettled neighbour with its updated cost and extended path, and let the heap keep everything ordered by cheapness.

In [ ]:
while pq:
    cost, u, path = heapq.heappop(pq)   # pop the cheapest-so-far node

    if u in settled:
        continue                        # already finalized via a cheaper route
    settled.add(u)

    if u == goal:
        print("cheapest cost S -> G:", cost)
        print("path:", " -> ".join(path))
        break

    # Relax each neighbour we haven't settled yet.
    for v, w in nbr[u]:
        if v not in settled:
            new_cost = cost + w
            new_path = path + [v]
            heapq.heappush(pq, (new_cost, v, new_path))

## Visualize the data & results

_Accounting for real driving distances, what is the cheapest route from the Ferry Building to the Castro?_

We'll run UCS over a small San Francisco map and chart the cheapest distance UCS discovers to *every* district.

### Step 1 — Run UCS to find the cheapest distance to every node

Here we use a slightly different UCS form that records the **best known cost** to each node in a `best` dictionary, rather than stopping at a single goal. A popped node whose cost is worse than its recorded best is stale and skipped. We relax a neighbour only when we find a strictly cheaper route to it.

In [ ]:
import heapq

# Edges are district pairs with driving distance in miles.
edges = [
    ("FB", "US", 1.2), ("FB", "CT", 0.9), ("CT", "NB", 0.7), ("US", "CT", 0.6),
    ("US", "SO", 1.0), ("SO", "MS", 1.1), ("US", "CC", 1.3), ("CC", "MS", 1.4),
    ("MS", "CA", 1.0), ("CC", "CA", 1.6), ("NB", "US", 1.1),
]

# Undirected adjacency list, same construction as before.
nbr = {}
for u, v, w in edges:
    nbr.setdefault(u, []).append((v, w))
    nbr.setdefault(v, []).append((u, w))

# UCS from the Ferry Building (FB), recording the cheapest cost to each node.
pq = [(0.0, "FB")]
best = {"FB": 0.0}

while pq:
    cost, u = heapq.heappop(pq)

    if cost > best.get(u, 1e9):
        continue                        # a cheaper route to u was already found

    for v, w in nbr[u]:
        new_cost = cost + w
        if new_cost < best.get(v, 1e9):
            best[v] = new_cost
            heapq.heappush(pq, (new_cost, v))

print("cheapest miles from FB:", {k: round(v, 1) for k, v in best.items()})

### Step 2 — Chart the cheapest distance to each district

Each bar is the cheapest total mileage UCS found from the Ferry Building (`FB`) to that district. The Castro (`CA`), our true destination, is highlighted in orange.

In [ ]:
# Show every district, ending with the Castro (CA) as the destination.
order = ["FB", "CT", "US", "NB", "SO", "CC", "MS", "CA"]
values = [round(best[n], 1) for n in order]
colors = ["#7ee787"] * 7 + ["#ffb454"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(order, values, color=colors)

# Label each bar with its cheapest mileage.
for i, v in enumerate(values):
    ax.text(i, v, str(v), ha="center", va="bottom")

ax.set_title("UCS cheapest miles to each district")
plt.show()